# 02 - Comparação dos Modelos

**Projeto:** Previsão de Evasão Escolar

Compara os três classificadores treinados no projeto — **XGBoost**, **LightGBM** e **KNN** — no mesmo conjunto de teste (split estratificado, `random_state=42`), usando a engenharia de atributos validada (`add_features=True`).

Métricas reportadas (média **macro**, pois as classes são desbalanceadas): **acurácia, precisão, recall e F1**. Ao final, a **matriz de confusão** de cada modelo.

> A lógica de treino vive em `src/models/`. Aqui apenas chamamos `run()` de cada módulo e comparamos os resultados.

> Antes de rodar: `python scripts/download_data.py` (gera `data/raw/dataset.csv`).

In [ ]:
import sys
import warnings
from pathlib import Path

# Permite importar o pacote src a partir do notebook
sys.path.append(str(Path.cwd().parent))

# Ruído inofensivo da BLAS (Accelerate/macOS) no matmul do PCA; resultados não mudam
warnings.filterwarnings('ignore', message='.*matmul.*', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, recall_score

from src.models import train_xgb, train_lgbm, train_knn
from src.models.evaluate import evaluate
from src.models.persistence import test_predictions

sns.set_theme(style='whitegrid')
pd.set_option('display.precision', 4)

## 1. Carga e avaliação dos três modelos

Este notebook **não re-treina**: ele carrega os modelos já treinados de `models/` (`xgboost.joblib`, `lightgbm.joblib`, `knn.joblib`) e os avalia no mesmo conjunto de teste (split estratificado, `random_state=42`).

Treine/atualize os modelos pela linha de comando (cada execução salva em `models/`):

```bash
python -m src.models.train_xgb    [--tune]
python -m src.models.train_lgbm   [--tune]
python -m src.models.train_knn    [--tune]
```

`--tune` faz a busca de hiperparâmetros via `GridSearchCV`. O modelo salvo é sempre o treinado com `add_features=True`. Para `test_predictions` reproduzir o split de teste idêntico, ele reusa o `get_xy` de cada módulo de treino.

> Se algum `.joblib` não existir, a célula abaixo levanta `FileNotFoundError` pedindo para treinar primeiro.

In [ ]:
modelos = [
    ('XGBoost', train_xgb),
    ('LightGBM', train_lgbm),
    ('KNN', train_knn),
]

metricas = {}
predicoes = {}  # nome -> (classes, y_test, y_pred) em rótulos originais

# Carrega cada modelo salvo de models/ e avalia no mesmo split de teste.
# (treine antes com: python -m src.models.train_xgb / train_lgbm / train_knn)
for nome, modulo in modelos:
    print(f'Carregando {nome} de {modulo.MODEL_PATH.name}...')
    le, y_test, y_pred = test_predictions(modulo.get_xy, modulo.MODEL_PATH)
    metricas[nome] = evaluate(y_test, y_pred)
    predicoes[nome] = (
        list(le.classes_),
        le.inverse_transform(y_test),
        le.inverse_transform(y_pred),
    )

print('Concluído.')

## 2. Tabela comparativa de métricas

Valores no conjunto de teste (precisão/recall/F1 em média macro).

In [ ]:
tabela = pd.DataFrame(metricas).T[['accuracy', 'precision', 'recall', 'f1']]
tabela = tabela.rename(columns={
    'accuracy': 'Acurácia',
    'precision': 'Precisão',
    'recall': 'Recall',
    'f1': 'F1',
})
tabela.index.name = 'Modelo'
tabela.sort_values('F1', ascending=False).style.format('{:.4f}').background_gradient(cmap='Greens')

In [ ]:
# Visualização das métricas em barras agrupadas
ax = tabela.plot(kind='bar', figsize=(9, 5), rot=0, colormap='viridis',
                 edgecolor='black', width=0.8)
ax.set_title('Comparação dos modelos por métrica (conjunto de teste)')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(loc='lower right', ncol=4)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 3. Matrizes de confusão

Linhas = classe real, colunas = classe prevista. A diagonal são os acertos.

In [ ]:
fig, axes = plt.subplots(1, len(modelos), figsize=(16, 4.5))

for ax, (nome, _) in zip(axes, modelos):
    classes, y_true, y_pred = predicoes[nome]
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'{nome}  (F1 macro = {metricas[nome]["f1"]:.3f})')
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
# Foco na classe Enrolled (a que mais errávamos): recall por modelo
enrolled = pd.Series({
    nome: recall_score(y_true, y_pred, labels=['Enrolled'], average='macro')
    for nome, (_, y_true, y_pred) in predicoes.items()
}, name='Recall Enrolled').sort_values(ascending=False)
print('Recall da classe Enrolled (fração dos Enrolled identificados):')
enrolled.round(3)

## 4. Importância das features (XGBoost)

Ganho (*gain*) médio de cada atributo nos splits do modelo salvo (`models/xgboost.joblib`): quanto maior, mais a feature contribuiu para separar as classes. Os nomes são traduzidos para português via `FEATURE_LABELS_PT`.

In [ ]:
import importlib
import src.features.build_features as bf
from src.models.persistence import load_bundle

# reimporta caso o módulo já estivesse carregado antes de FEATURE_LABELS_PT existir
importlib.reload(bf)
FEATURE_LABELS_PT = bf.FEATURE_LABELS_PT

# Importância por GANHO (gain): ganho médio que cada feature trouxe nos splits
# do modelo salvo em models/xgboost.joblib (só aparecem features de fato usadas).
xgb_model = load_bundle(train_xgb.MODEL_PATH)['model']
gain = xgb_model.get_booster().get_score(importance_type='gain')

imp = (pd.Series(gain, name='gain')
       .sort_values(ascending=False)
       .head(15)
       .iloc[::-1])  # menor -> maior, para o barh ficar com a maior no topo

# Traduz os rótulos para português (fallback no nome original se faltar)
imp.index = [FEATURE_LABELS_PT.get(c, c) for c in imp.index]

fig, ax = plt.subplots(figsize=(9, 6))
imp.plot(kind='barh', color='#3274A1', edgecolor='black', ax=ax)
ax.set_title('XGBoost — Top 15 atributos por importância (ganho)')
ax.set_xlabel('Ganho médio')
ax.set_ylabel('Atributo')
ax.bar_label(ax.containers[0], fmt='%.1f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Conclusão

- Os modelos de **boosting (XGBoost e LightGBM)** lideram em todas as métricas — capturam bem as interações não-lineares do dado tabular. O **KNN** fica atrás mesmo com escala + PCA, por causa da alta dimensionalidade (one-hot das nominais).
- A classe **Enrolled** era a mais difícil para todos os modelos: fica no meio do caminho entre evadir e formar e é a menos frequente, então concentrava os erros.